In [ ]:
import os
import re
import math
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

Set styling for visualisations

In [ ]:
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

##Step 1: Data Acquisition (Web scraping)

In [ ]:
RATING_MAP = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

BASE_URL = "https://books.toscrape.com/catalogue/"
START_URL = "https://books.toscrape.com/catalogue/page-1.html"

def scrape_catalogue(max_pages=15):
    
    books = []
    current_url = START_URL
    page_count = 0

    while current_url and page_count < max_pages:
        page_count += 1
        response = requests.get(current_url)
        if response.status_code != 200:
            print(f"Failed to retrieve page {page_count}")
            break

        soup = BeautifulSoup(response.content, "html.parser")
        articles = soup.find_all("article", class_="product_pod")

        for article in articles:
            # Title
            title = article.h3.a["title"]
            
            # Price
            price_text = article.find("p", class_="price_color").text
            
            # Star Rating
            rating_class = article.find("p", class_="star-rating")["class"]
            rating_word = [c for c in rating_class if c != "star-rating"][0]
            
            # Availability status
            availability = article.find("p", class_="instock availability").text.strip()
            
            # Relative link to detail page to fetch Category/Genre
            detail_rel_path = article.h3.a["href"]
            detail_url = BASE_URL + detail_rel_path.replace("catalogue/", "")
            
            books.append({
                "title": title,
                "price_raw": price_text,
                "rating_raw": rating_word,
                "availability_raw": availability,
                "detail_url": detail_url
            })

        # Pagination
        next_button = soup.find("li", class_="next")
        if next_button:
            next_page_rel = next_button.a["href"]
            current_url = BASE_URL + next_page_rel
        else:
            current_url = None

    print(f"Scraped {len(books)} books across {page_count} pages.")
    return pd.DataFrame(books)

# Execute scraping
raw_df = scrape_catalogue(max_pages=15) 



Step 2: Data cleaning & Feature engineering

In [ ]:
def clean_dataset(df):
    cleaned_df = df.copy()

    # 1. Convert Price to float (remove '£' symbol)
    cleaned_df["price_gbp"] = (
        cleaned_df["price_raw"]
        .str.replace("£", "", regex=False)
        .str.replace("Â", "", regex=False)
        .astype(float)
    )

    # 2. Map Rating Word to Integer (1 to 5)
    cleaned_df["rating"] = cleaned_df["rating_raw"].map(RATING_MAP).fillna(0).astype(int)

    # 3. Clean Availability into Binary Flag
    cleaned_df["in_stock"] = cleaned_df["availability_raw"].apply(
        lambda x: 1 if "in stock" in x.lower() else 0
    )

    # 4. Feature Engineering: Title Length & Word Count
    cleaned_df["title_char_length"] = cleaned_df["title"].apply(len)
    cleaned_df["title_word_count"] = cleaned_df["title"].apply(lambda x: len(x.split()))

    # Select and reorder clean columns
    final_cols = ["title", "price_gbp", "rating", "in_stock", "title_char_length", "title_word_count"]
    cleaned_df = cleaned_df[final_cols].dropna().drop_duplicates()
    
    return cleaned_df

clean_df = clean_dataset(raw_df)
clean_df.to_csv("scraped_books_clean.csv", index=False)
print("Data clean & saved to 'scraped_books_clean.csv'.")

Step 3: Data Visualization & EDA 

In [ ]:


# 1. Price Distribution
plt.figure(figsize=(8, 4))
sns.histplot(clean_df["price_gbp"], kde=True, color="skyblue")
plt.title("Distribution of Book Prices (£)")
plt.xlabel("Price (£)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig("price_distribution.png")
plt.close()

# 2. Price by Star Rating Boxplot
plt.figure(figsize=(8, 4))
sns.boxplot(x="rating", y="price_gbp", data=clean_df, palette="Blues")
plt.title("Book Price Distribution Across Star Ratings")
plt.xlabel("Star Rating (1-5)")
plt.ylabel("Price (£)")
plt.tight_layout()
plt.savefig("price_vs_rating.png")
plt.close()

# 3. Correlation Matrix
plt.figure(figsize=(6, 5))
sns.heatmap(clean_df[["price_gbp", "rating", "title_char_length", "title_word_count"]].corr(), 
            annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.savefig("correlation_matrix.png")
plt.close()

print("All EDA visualisations successfully generated and saved as PNGs")

Step 4: Predictive Modelling (Linear regression)

In [1]:
# Features and Target selection
features = ["rating", "title_char_length", "title_word_count"]
X = clean_df[features]
y = clean_df["price_gbp"]

# Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit Model
model = LinearRegression()
model.fit(X_train, y_train)

# Predictions & Evaluation
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("--- Model Performance ---")
print(f"MAE:  £{mae:.2f}")
print(f"RMSE: £{rmse:.2f}")
print(f"R² Score: {r2:.4f}")

# Display Coefficients
coef_df = pd.DataFrame({"Feature": features, "Coefficient": model.coef_})
coef_df.loc[len(coef_df)] = ["Intercept", model.intercept_]
print("\n--- Model Coefficients ---")
print(coef_df.to_string(index=False))

NameError: name 'clean_df' is not defined